# 43. Image-Mask Dataset과 DataLoader 구현

이 노트북은 image와 mask를 함께 반환하는 PyTorch Dataset 구조를 다룹니다.

이번 노트북의 목표는 다음과 같습니다.

- segmentation Dataset의 `__getitem__` 반환 형식을 이해합니다.
- image와 mask에 서로 다른 변환이 필요하다는 점을 확인합니다.
- DataLoader batch shape를 점검합니다.

In [ ]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

torch.manual_seed(0)
np.random.seed(0)

## 43-1. Synthetic segmentation dataset

파일 입출력 대신 작은 합성 데이터를 만들어 Dataset 구조를 연습합니다. 실제 프로젝트에서는 이 부분이 image path와 mask path를 읽는 코드로 바뀝니다.

In [ ]:
class SyntheticSegmentationDataset(Dataset):
    def __init__(self, length=16, size=96, num_classes=3):
        self.length = length
        self.size = size
        self.num_classes = num_classes

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        h = w = self.size
        yy, xx = np.mgrid[:h, :w]

        image = np.zeros((3, h, w), dtype=np.float32)
        mask = np.zeros((h, w), dtype=np.int64)

        cx = 25 + (idx * 7) % 45
        cy = 28 + (idx * 5) % 40
        circle = (xx - cx) ** 2 + (yy - cy) ** 2 < 16 ** 2
        rect = (xx > 48) & (xx < 82) & (yy > 30 + idx % 10) & (yy < 72)

        image[:] = 0.55
        image[0, circle] = 0.95
        image[1, rect] = 0.85
        image += np.random.normal(0, 0.03, size=image.shape).astype(np.float32)

        mask[circle] = 1
        mask[rect] = 2

        image = np.clip(image, 0, 1)
        return torch.from_numpy(image), torch.from_numpy(mask)


dataset = SyntheticSegmentationDataset()
image, mask = dataset[0]
print("one image:", image.shape, image.dtype)
print("one mask:", mask.shape, mask.dtype)
print("mask ids:", torch.unique(mask))

## 43-2. DataLoader batch 확인

모델에 들어가는 batch는 보통 다음 shape를 갖습니다.

```text
images: B, 3, H, W
masks:  B, H, W
```

In [ ]:
loader = DataLoader(dataset, batch_size=4, shuffle=True)
images, masks = next(iter(loader))

print("batch images:", images.shape)
print("batch masks:", masks.shape)

## 43-3. Image와 mask resize의 차이

image는 bilinear resize를 사용할 수 있지만, mask는 class id가 깨지지 않도록 nearest resize를 사용해야 합니다.

In [ ]:
small_images = F.interpolate(images, size=(64, 64), mode="bilinear", align_corners=False)
small_masks = F.interpolate(masks.unsqueeze(1).float(), size=(64, 64), mode="nearest").squeeze(1).long()

print("resized images:", small_images.shape, small_images.dtype)
print("resized masks:", small_masks.shape, small_masks.dtype)
print("mask ids after nearest:", torch.unique(small_masks))

## 43-4. 실제 파일 Dataset으로 바꿀 때

실제 데이터셋에서는 다음 항목을 확인합니다.

- image path와 mask path가 같은 sample을 가리키는가?
- image는 `float32`, mask는 `long`인가?
- image normalization이 pretrained model의 기준과 맞는가?
- mask resize가 nearest로 처리되는가?

## 정리

- segmentation Dataset은 image tensor와 mask tensor를 함께 반환합니다.
- batch shape는 `B, 3, H, W`와 `B, H, W`가 기본입니다.
- 다음 노트북 `44_Train_Validation_Loop_구현.ipynb`에서는 이 DataLoader로 학습 loop를 구성합니다.